# Verified outcomes for a LangGraph payment tool

This walkthrough runs the same account-free sandbox flow as `graph.py`. It compiles criteria for 10 records at $1.00 each, submits evidence where 8 records pass and 2 are rejected, verifies the outcome, pauses before the payment stub, and resumes for the exact earned amount. No real money moves.

## Build the graph

The example assumes LangGraph 1.2.11. Its in-memory checkpointer and thread ID let `interrupt()` save and resume this short-lived run.

In [ ]:
import json
import subprocess
import uuid
from decimal import Decimal

from langgraph.types import Command
from graph import build_graph

graph = build_graph()
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

## Compile, submit evidence, and verify

The first invocation runs `buy_records` and `verify`, then pauses in `payment_gate`. The interrupt payload comes from the signed receipt already stored in graph state.

In [ ]:
paused = graph.invoke({}, config=config)
outcome = paused["__interrupt__"][0].value
outcome

## Resume from the verified outcome

The runner applies its policy to the verified result and binds the resume command to the receipt ID. The payment tool receives the decimal string from `amounts.earned`.

In [ ]:
assert outcome["result"] in {"pass", "partial"}
assert Decimal(outcome["earned"]) >= 0
final = graph.invoke(
    Command(resume={"receipt_id": outcome["receipt_id"]}),
    config=config,
)
final["payment_message"]

## Verify the receipt signature locally

Save the receipt and run the small Node.js verifier against the published Spoolis trust set. Signature validity proves authenticity and integrity, not the truth of the evidence or payment authorization.

In [ ]:
with open("outcome-receipt.json", "w", encoding="utf-8") as receipt_file:
    json.dump(final["receipt"], receipt_file, indent=2)
    receipt_file.write("\n")

signature_check = subprocess.run(
    ["node", "verify_receipt.mjs", "outcome-receipt.json"],
    check=True,
    capture_output=True,
    text=True,
)
result = {
    "earned": f"{final['earned']} {final['asset']}",
    "signature": json.loads(signature_check.stdout),
}
result